# 01 — Data Exploration

Initial exploratory analysis of the OkCupid profiles dataset (~60K rows, 31 columns).
Goals:
- Understand schema and value distributions
- Quantify missingness per column
- Identify variables worth deeper analysis

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.cleaning import initial_quality

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

In [ ]:
df = pd.read_csv("../data/raw/okcupid_profiles.csv")
print(f"Loaded {len(df):,} profiles, {df.shape[1]} columns")
df.head()

## Schema overview

In [ ]:
df.dtypes

## Missingness

In [ ]:
report = initial_quality(df)
print(f"Rows:                       {report['n_rows']:,}")
print(f"Columns:                    {report['n_cols']}")
print(f"% rows fully complete:      {report['pct_rows_fully_complete']:.1%}")
print(f"% rows with core fields:    {report['pct_rows_core_complete']:.1%}")

In [ ]:
missing = pd.Series(report['missing_by_col']).sort_values()
fig, ax = plt.subplots(figsize=(8, 8))
missing.plot.barh(ax=ax, color='#4A90E2')
ax.set_xlabel('Missing fraction')
ax.set_title('Missing values per column')
ax.axvline(0.5, ls='--', color='red', alpha=0.5, label='50%')
ax.legend()
plt.tight_layout()
plt.show()

## Numeric distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['age'].plot.hist(bins=50, ax=axes[0], color='#4A90E2')
axes[0].set_title('Age')
axes[0].set_xlabel('Years')

df['height'].dropna().plot.hist(bins=50, ax=axes[1], color='#7ED321')
axes[1].set_title('Height (inches)')

income_clean = df.loc[df['income'] > 0, 'income']
income_clean.plot.hist(bins=30, ax=axes[2], color='#F5A623')
axes[2].set_title('Income (disclosed only)')
axes[2].set_xlabel('USD')

plt.tight_layout()
plt.show()

print(f"Income disclosure rate: {(df['income'] > 0).mean():.1%}")

## Categorical breakdowns

In [ ]:
for col in ['sex', 'orientation', 'status', 'drinks', 'smokes', 'drugs']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False, normalize=True).head(8).to_string())

## Essay length distribution

In [ ]:
essay_cols = [f'essay{i}' for i in range(10)]
essay_lengths = df[essay_cols].fillna('').apply(lambda s: s.str.len())

fig, ax = plt.subplots(figsize=(10, 4))
essay_lengths.median().plot.bar(ax=ax, color='#9013FE')
ax.set_title('Median essay length per essay field')
ax.set_ylabel('Characters')
plt.tight_layout()
plt.show()

print(f"\nBio (essay0) length stats:")
print(essay_lengths['essay0'].describe().to_string())

## Takeaways

Key observations to inform downstream cleaning:

- **High missingness columns**: `offspring`, `diet`, `religion`, `sign` (>30% missing) — need imputation strategy
- **Income**: `-1` is a sentinel for "not disclosed" — must be recoded to NaN before any numeric work
- **Age**: clean, plausible range 18-110
- **Height**: has missing values + a few implausible outliers — needs clipping
- **Essays**: highly variable length; `essay0` ("About me") is the canonical bio field